In [1]:
import pandas as pd
import os
from statsmodels.tsa.ardl import ARDL
import statsmodels.api as sm
import numpy as np
from scipy.stats import f

# ============================================================
# 1. Load dữ liệu đã xử lý
# ============================================================
data_file = "../data/processed/data_processed.csv"
data_processed = pd.read_csv(data_file)

# ============================================================
# 2. Cấu hình 3 model song song
# ============================================================
models = {
    "Model_1_TB": {
        "dep_var": "ln_TB",
        "exog_vars": ["ln_RER_pos","ln_RER_neg","ln_FDI","IPI_VN","IPI_World_GR_diff","ln_M2_diff","ln_WTI","COVID"],
        "lag_dep": 5,
        "lag_exog": [0,6,0,6,0,0,0,0]
    },
    "Model_2_EX": {
        "dep_var": "ln_EX",
        "exog_vars": ["ln_RER_pos","ln_RER_neg","ln_FDI","IPI_VN","IPI_World_GR_diff","ln_M2_diff","ln_WTI","COVID"],
        "lag_dep": 6,
        "lag_exog": [3,1,6,6,5,2,0]
    },
    "Model_3_IM": {
        "dep_var": "ln_IM",
        "exog_vars": ["ln_RER_pos","ln_RER_neg","ln_FDI","IPI_VN","IPI_World_GR_diff","ln_M2_diff","ln_WTI","COVID"],
        "lag_dep": 3,
        "lag_exog": [3,1,2,2,4,3,0]
    }
}

# ============================================================
# 3. Tạo folder results
# ============================================================
output_folder = "../results"

# ============================================================
# 4. Hàm tạo lag
# ============================================================
def create_lags(df, var_list, lag_list):
    df_lagged = df.copy()
    for var, lag in zip(var_list, lag_list):
        if lag > 0:
            for l in range(1, lag+1):
                df_lagged[f"{var}_L{l}"] = df_lagged[var].shift(l)
    return df_lagged

# ============================================================
# 5. Chạy Bounds Test cho từng model
# ============================================================
bounds_results = []

for model_name, spec in models.items():
    dep = spec["dep_var"]
    exogs = [v for v in spec["exog_vars"] if v in data_processed.columns]

    df_model = data_processed[[dep] + exogs].dropna()
    df_model = create_lags(df_model, exogs, spec["lag_exog"])

    # Xử lý NaN/Inf do lag
    df_model.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_model.dropna(inplace=True)

    # Chuẩn bị danh sách exog lagged
    exog_lagged_vars = [col for col in df_model.columns if col != dep]

    # Step 1: Unrestricted ECM model (OLS)
    Y_unres = df_model[dep]
    X_unres = sm.add_constant(df_model[exog_lagged_vars])
    res_unres = sm.OLS(Y_unres, X_unres).fit()

    # Step 2: Restricted model (ép tất cả hệ số long-run = 0)
    # restricted chỉ giữ hằng số
    X_restr = sm.add_constant(pd.DataFrame(index=df_model.index))
    res_restr = sm.OLS(Y_unres, X_restr).fit()

    # Step 3: Tính F-statistic
    k = len(exog_lagged_vars)
    n = len(Y_unres)
    SSR_unres = res_unres.ssr
    SSR_restr = res_restr.ssr
    df_num = k
    df_den = n - k - 1

    F_stat = ((SSR_restr - SSR_unres)/df_num) / (SSR_unres / df_den)

    # Step 4: Tính P-value
    p_val = 1 - f.cdf(F_stat, df_num, df_den)

    bounds_results.append({
        "Model": model_name,
        "Dependent": dep,
        "F_statistic": F_stat,
        "P_value": p_val
    })

    print("\n" + "="*60)
    print(f"Bounds Test for {model_name} ({dep})")
    print("="*60)
    print(f"F-statistic: {F_stat:.4f}, P-value: {p_val:.4f}")

# ============================================================
# 6. Xuất kết quả ra CSV
# ============================================================
output_file = os.path.join(output_folder, "03_bound_test.csv")
pd.DataFrame(bounds_results).to_csv(output_file, index=False)
print(f"\nBounds Test results exported to: {output_file}")


Bounds Test for Model_1_TB (ln_TB)
F-statistic: 1.0177, P-value: 0.4477

Bounds Test for Model_2_EX (ln_EX)
F-statistic: 34.4138, P-value: 0.0000

Bounds Test for Model_3_IM (ln_IM)
F-statistic: 31.9203, P-value: 0.0000

Bounds Test results exported to: ../results\03_bound_test.csv
